# SFT on MBPP (Qwen2.5-Coder-3B, LoRA, T4)

Supervised fine-tuning of **Qwen2.5-Coder-3B-Instruct** on the MBPP subset of `final_dataset_v2.csv` using **QLoRA** (4-bit + LoRA). Designed for Google Colab with **T4 GPU**.

Outputs LoRA adapters in PEFT format and a FedAvg-ready `lora_state_dict.pt`.

In [1]:
# Check GPU (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi

Thu Mar 12 05:42:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q "transformers>=4.36" "peft>=0.7" "bitsandbytes>=0.41" "trl>=0.7,<0.20" "datasets" "accelerate" "pandas"
# Restart runtime after install if needed, then run the rest of the notebook.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 39.3 MB/s eta 0:00:00


## Data loading

In [3]:
from google.colab import files
import pandas as pd

# Option A: Upload final_dataset_v2.csv when prompted (or use copy from FED/MBPP/)
uploaded = files.upload()
CSV_PATH = list(uploaded.keys())[0]
print("Using CSV:", CSV_PATH)

# Option B (comment out Option A and set path manually):
# CSV_PATH = "/content/drive/MyDrive/.../final_dataset_v2.csv"  # after mounting Drive

df = pd.read_csv(CSV_PATH)
print("Total rows:", len(df))

Saving final_dataset_v2.csv to final_dataset_v2.csv
Using CSV: final_dataset_v2.csv
Total rows: 1491


In [4]:
df_mbpp = df[df["dataset"] == "mbpp"].copy()
df_mbpp = df_mbpp.dropna(subset=["prompt", "canonical_solution"])
df_mbpp["canonical_solution"] = df_mbpp["canonical_solution"].astype(str).str.strip()
df_mbpp = df_mbpp[df_mbpp["canonical_solution"].str.len() > 0]
assert len(df_mbpp) > 0, f"Expected MBPP rows, got {len(df_mbpp)}"
print("MBPP rows:", len(df_mbpp))

MBPP rows: 120


## Parse prompt and build chat messages

In [5]:
SEP = "\n\nUser: "

def row_to_messages(row):
    prompt_str = str(row["prompt"]).strip()
    idx = prompt_str.find(SEP)
    if idx == -1:
        import warnings
        warnings.warn(f"Row {row.get('task_id')}: no '\\n\\nUser: ' found; using whole prompt as user.")
        system_content = "You are an expert Python developer. Complete the function provided by the user."
        user_content = prompt_str.replace("System: ", "", 1).strip()
    else:
        system_content = prompt_str[:idx].replace("System: ", "", 1).strip()
        user_content = prompt_str[idx + len(SEP):].strip()
    solution = str(row["canonical_solution"]).strip()
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": solution},
    ]

messages_list = [row_to_messages(row) for _, row in df_mbpp.iterrows()]
print("Built", len(messages_list), "message lists.")

Built 120 message lists.


In [6]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"messages": messages_list})
print(train_dataset)

Dataset({
    features: ['messages'],
    num_rows: 120
})


## Model and tokenizer

In [7]:
from transformers import AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded.


In [8]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print("Model loaded in 4-bit.")

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded in 4-bit.


## LoRA (PEFT) configuration

**FedAvg**: Use this exact config on all clients so state dict keys and shapes match when averaging.

In [9]:
from peft import LoraConfig, get_peft_model

LORA_R = 16
LORA_ALPHA = 32
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


## Training

In [10]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "./sft_mbpp_output/lora_adapters"

# Completion-only loss: mask prompt tokens (try DataCollatorForCompletionOnlyLM if available)
try:
    from trl import DataCollatorForCompletionOnlyLM
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
except ImportError:
    try:
        from trl.extras import DataCollatorForCompletionOnlyLM
        response_template = "<|im_start|>assistant\n"
        collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)
    except ImportError:
        collator = None

training_args = SFTConfig(
    output_dir="./sft_mbpp_output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="epoch",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    save_total_limit=1,
    max_seq_length=2048,
    packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [11]:
def formatting_func(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    formatting_func=formatting_func,
    processing_class=tokenizer,
)
if collator is not None:
    trainer_kwargs["data_collator"] = collator

trainer = SFTTrainer(**trainer_kwargs)
print("SFTTrainer created.")

Applying formatting function to train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/120 [00:00<?, ? examples/s]

SFTTrainer created.


In [12]:
# Optional: quick sanity run (comment out after verifying)
# trainer.train(max_steps=2)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5,0.875979
10,0.464139
15,0.369147
20,0.318463
25,0.264357
30,0.303692
35,0.258919
40,0.207409
45,0.153711


TrainOutput(global_step=45, training_loss=0.35731276671091716, metrics={'train_runtime': 506.6591, 'train_samples_per_second': 0.711, 'train_steps_per_second': 0.089, 'total_flos': 1027128698830848.0, 'train_loss': 0.35731276671091716})

## Save adapter (PEFT format)

In [13]:
import os
os.makedirs(ADAPTER_DIR, exist_ok=True)
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Saved adapter and tokenizer to", ADAPTER_DIR)
print("Files:", os.listdir(ADAPTER_DIR))

Saved adapter and tokenizer to ./sft_mbpp_output/lora_adapters
Files: ['README.md', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'adapter_config.json']


## Get and print LoRA adapters (FedAvg-ready, visible as output)

In [14]:
lora_state = {
    k: v.detach().cpu().clone()
    for k, v in model.named_parameters()
    if v.requires_grad
}

print("=== LoRA adapter structure (name -> shape) ===")
for name, tensor in lora_state.items():
    print(f"  {name}: {tensor.shape}")

print("\n=== Per-parameter summary (min, max, norm) ===")
for name, tensor in lora_state.items():
    t = tensor.float()
    print(f"  {name}: min={t.min().item():.4f}, max={t.max().item():.4f}, norm={t.norm().item():.4f}")

total_params = sum(p.numel() for p in lora_state.values())
print(f"\n=== Summary ===")
print(f"  Number of LoRA parameters: {len(lora_state)}")
print(f"  Total elements: {total_params}")
print("  Parameter names (for FedAvg):", list(lora_state.keys()))

=== LoRA adapter structure (name -> shape) ===
  base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight: torch.Size([256, 16])
  base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight: torch.Size([2048, 16])
  base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight: torch.Size([16, 2048])
  base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight: torch.Size([11

**FedAvg**: Use `lora_state_dict.pt` below for aggregation. Load it with `torch.load(...)` and average the tensors with other clients' LoRA state dicts (same keys and shapes).

In [15]:
lora_pt_path = os.path.join(ADAPTER_DIR, "lora_state_dict.pt")
torch.save(lora_state, lora_pt_path)
print("Saved FedAvg-ready LoRA state dict to", lora_pt_path)
print("File size (MB):", os.path.getsize(lora_pt_path) / (1024 * 1024))

# Optional: copy to Google Drive to keep after session
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r ./sft_mbpp_output /content/drive/MyDrive/

Saved FedAvg-ready LoRA state dict to ./sft_mbpp_output/lora_adapters/lora_state_dict.pt
File size (MB): 114.35906887054443


In [16]:
from google.colab import files
files.download(lora_pt_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
!zip -r sft_mbpp_output.zip ./sft_mbpp_output

  adding: sft_mbpp_output/ (stored 0%)
  adding: sft_mbpp_output/README.md (deflated 43%)
  adding: sft_mbpp_output/checkpoint-45/ (stored 0%)
  adding: sft_mbpp_output/checkpoint-45/README.md (deflated 65%)
  adding: sft_mbpp_output/checkpoint-45/chat_template.jinja (deflated 71%)
  adding: sft_mbpp_output/checkpoint-45/scheduler.pt (deflated 62%)
  adding: sft_mbpp_output/checkpoint-45/rng_state.pth (deflated 26%)
  adding: sft_mbpp_output/checkpoint-45/trainer_state.json (deflated 68%)
  adding: sft_mbpp_output/checkpoint-45/optimizer.pt (deflated 8%)
  adding: sft_mbpp_output/checkpoint-45/adapter_model.safetensors (deflated 8%)
  adding: sft_mbpp_output/checkpoint-45/tokenizer_config.json (deflated 59%)
  adding: sft_mbpp_output/checkpoint-45/tokenizer.json (deflated 81%)
  adding: sft_mbpp_output/checkpoint-45/training_args.bin (deflated 53%)
  adding: sft_mbpp_output/checkpoint-45/adapter_config.json (deflated 58%)
  adding: sft_mbpp_output/lora_adapters/ (stored 0%)
  adding: s

In [18]:
from google.colab import files
files.download("sft_mbpp_output.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>